# Data Preparation

Der erste Schritt normalisiert die ROhdaten und reichert sie mit query-relevanten Daten an. Das Anreichern wird mit einem LLM durchgeführt. Die Daten prüfe ich nur stichprobenartig, sie gelten erstmal als so wahr.

- Rohdaten: Aus einem Vibecoding-Projekt.
- LLM: Mistral 

In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

with open('../data/raw/products_raw.json', 'r') as f:
     products = json.load(f)

# Evaluation
products = products[:3]

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key)

def agent_request(system_promt, schema, content):
            
    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

## Beschreibungen

Die Beschreibungen sollen so gegliedert sein, dass jeder Absatz ein Thema behandelt und das Produktname und Hersteller genannt wird. Werbliche Texte sollen entfernt werden. Die Antwort wird als JSON erwartet, es wird die dazu _response_format_ der API genutzt.

In [ ]:
with open('../data/promts/descs_agent.md', 'r') as f:
    descs_promt = f.read()

with open('../data/promts/descs_schema.json', 'r')as f:
    descs_schema = json.load(f)

for product in tqdm(products, total=len(products)):

    desc_response = agent_request(descs_promt, descs_schema, product['description'])
    product['desc_documents'] = json.loads(desc_response.choices[0].message.content)
    product['desc_usage'] = desc_response.usage.model_dump()


## Technische Daten

Bei den technischen Daten werden prinzipiell die selben Daten wie zur Beschreibung hinzugefügt, allerdings pro Objekt.

In [ ]:
with open('../data/promts/specs_agent.md', 'r') as f:
    specs_promt = f.read()

with open('../data/promts/specs_schema.json', 'r') as f:
    specs_schema = json.load(f)

for product in tqdm(products, total=len(products), desc="Products"):

    specs_serialaized = json.dumps(product['specs'])
    
    specs_response = agent_request(specs_promt, specs_schema, specs_serialaized)
    product['specs_documents'] = json.loads(specs_response.choices[0].message.content)
    product['specs_usage'] = specs_response.usage.model_dump()

Products: 100%|██████████| 3/3 [02:32<00:00, 50.99s/it]


## Speichern

In [36]:
with open("../data/processed/products_enriched.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

## Evaluation

Einmal nachsehen wie lange die Documents geworden sind und ob alle gefüllt wurden

In [ ]:
with open('../data/processed/products_enriched.json', 'r', encoding='utf-8') as f:
    evaldata = json.load(f)

specs_chunks = []
descs_chunks = []

for product in evaldata:
    product_id = product['id']

    for i, desc in enumerate(product.get('specs_documents', [])):
        specs_chunks.append({
            'text': desc,
            'length': len(desc)
        })

df = pd.DataFrame(evaldata)
# Fortsetzung folgt
